In [155]:
from unsloth import FastLanguageModel
from datasets import load_dataset, disable_caching
from transformers import TrainingArguments, EarlyStoppingCallback
from trl import SFTTrainer
import torch
import pandas as pd

disable_caching()
torch.cuda.empty_cache()

In [156]:
MODEL_NAME = "unsloth/llama-2-7b-bnb-4bit"
DATA_PATH = "data/flipped_dataset.csv"
SEED = 42
MAX_SEQ_LENGTH = 256
SUBSET_SIZE = 20000 # 500, 1000, 5000, 10K, 20K, 100K

In [ ]:
dataset = load_dataset("csv", data_files=DATA_PATH)["train"]

# Shuffle and Subset
dataset = dataset.shuffle(seed=SEED)
dataset = dataset.select(range(min(SUBSET_SIZE, len(dataset))))

split = dataset.train_test_split(test_size=0.2, seed=SEED)
train_dataset = split["train"]
eval_dataset = split["test"]

In [158]:
def create_text_column(example):
    # Combine all candidate fields into a single string
    prompt = (
        f"Below is an instruction that describes a fair hiring decision task.\n\n"
        f"### Instruction:\n"
        f"You are an unbiased AI hiring assistant. \n"
        f"Make a decision based ONLY on qualifications. \n"
        f"Ignore gender and any demographic attributes.\n\n"
        f"Given the following candidate profile, determine whether the candidate should be hired.\n\n"
        f"Education Level: {example['education_level']}\n"
        f"Specialization Domain: {example['specialization_domain']}\n"
        f"Has Certification: {example['has_certification']}\n"
        f"Skill Count: {example['skill_count']}\n"
        f"Tech Skill Count: {example['tech_skill_count']}\n"
        f"Soft Skill Count: {example['soft_skill_count']}\n"
        f"Education Job Match: {example['education_job_match']}\n"
        f"Highest Qualification Level: {example['highest_qualification_level']}\n"
        f"Gender: {example['Gender']}\n\n"
        f"### Response:\n"
        f"{example['is_employed']}"
    )
    return {"text": prompt}

train_dataset = train_dataset.map(create_text_column)
eval_dataset = eval_dataset.map(create_text_column)

Map: 100%|██████████| 4000/4000 [00:00<00:00, 5526.94 examples/s]


In [159]:
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=None,
    load_in_4bit=True,
    device_map = "auto",
     max_memory = {
        0: "9GB",
        "cpu": "48GB"
    }
)

==((====))==  Unsloth 2026.3.3: Fast Llama patching. Transformers: 4.57.1.
   \\   /|    NVIDIA GeForce RTX 3080. Num GPUs = 1. Max memory: 10.0 GB. Platform: Windows.
O^O/ \_/ \    Torch: 2.10.0+cu130. CUDA: 8.6. CUDA Toolkit: 13.0. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


In [160]:
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha=16,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing=True,
    random_state=SEED,
    max_seq_length=MAX_SEQ_LENGTH,
)

In [161]:
EOS_TOKEN = tokenizer.eos_token

def format_prompt(example):
    prompt = example["text"] + EOS_TOKEN
    tokenized = tokenizer(prompt, truncation=True, max_length=MAX_SEQ_LENGTH)
    tokenized["labels"] = tokenized["input_ids"].copy()  # supervised fine-tuning
    return tokenized

In [ ]:
training_args = TrainingArguments(
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    num_train_epochs=3,
    warmup_steps=10,
    logging_steps=10,
    eval_strategy="steps",
    eval_steps=50,
    save_steps=50,
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    output_dir="outputs",
    run_name=f"llama2_bias_finetune_{SUBSET_SIZE}",
    optim="adamw_8bit",
    fp16=not torch.cuda.is_bf16_supported(),
    bf16=torch.cuda.is_bf16_supported(),
    lr_scheduler_type="linear",
    seed=SEED,
    report_to="none"
)

In [163]:
trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    dataset_text_field="text",
    max_seq_length=MAX_SEQ_LENGTH,
    args=training_args,
    formatting_func=format_prompt,
    callbacks=[EarlyStoppingCallback(
        early_stopping_patience=10,
        early_stopping_threshold=0.001
    )]
)

Unsloth: Tokenizing ["text"]: 100%|██████████| 4000/4000 [00:00<00:00, 6438.85 examples/s]


In [164]:
trainer.train()

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 16,000 | Num Epochs = 3 | Total steps = 12,000
O^O/ \_/ \    Batch size per device = 1 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (1 x 4 x 1) = 4
 "-____-"     Trainable parameters = 39,976,960 of 6,778,392,576 (0.59% trained)


Step,Training Loss,Validation Loss
50,0.164100,0.108510
100,0.058000,0.055596
150,0.052900,0.053157
200,0.044500,0.050634
250,0.053400,0.051441
300,0.052300,0.049097
350,0.050300,0.047138
400,0.042600,0.045881
450,0.050500,0.043144
500,0.048600,0.042903


TrainOutput(global_step=1150, training_loss=0.08881885686646337, metrics={'train_runtime': 23835.2341, 'train_samples_per_second': 2.014, 'train_steps_per_second': 0.503, 'total_flos': 2.8212730818994176e+16, 'train_loss': 0.08881885686646337, 'epoch': 0.2875})

In [ ]:
from huggingface_hub import login
from dotenv import load_dotenv
import os

load_dotenv()
login(token=os.getenv("HUGGINGFACE_TOKEN"))

SAVE_PATH = f"llama2_bias_finetune_{SUBSET_SIZE}"

trainer.save_model(SAVE_PATH)
tokenizer.save_pretrained(SAVE_PATH)

model.push_to_hub(f"moosejuice13/{SAVE_PATH}")
tokenizer.push_to_hub(f"moosejuice13/{SAVE_PATH}")